# NumPy Interview Prep: Questions, Answers & Practice

**How to use this notebook**
1. Read the question and try to answer it *out loud* first.
2. Read the short answer, then run the code cell to see it in action.
3. For the *Predict the Output* and *Coding Challenges* sections, try on your own before looking at the solution.

**Sections**
1. Fundamentals (why NumPy, dtypes)
2. Array creation
3. Array attributes
4. Indexing, slicing, masking, view vs copy
5. Broadcasting
6. Reshaping & transposing
7. Math, aggregation & ufuncs
8. Joining & splitting
9. Sorting, searching & set operations
10. Linear algebra
11. Random numbers
12. NaN & inf handling
13. Memory, performance & I/O
14. Predict the Output (tricky questions)
15. Coding challenges with solutions
16. Rapid-fire theory cheat sheet

In [ ]:
import numpy as np
import time
import sys

print("NumPy version:", np.__version__)

---
# 1. Fundamentals

### Q1. What is NumPy and why use it instead of Python lists?

**Answer:** NumPy is the core library for numerical computing in Python. Its main object is the `ndarray`, an N-dimensional array that stores elements of the **same type** in a **contiguous block of memory**.

Why it beats lists:
- **Speed**: operations run in compiled C code, not in the Python interpreter loop.
- **Memory**: no per-element Python object overhead, and elements are packed tightly.
- **Vectorization**: you write `a * 2` instead of a loop.
- **Rich toolkit**: broadcasting, linear algebra, random numbers, FFT, statistics.

In [ ]:
size = 1_000_000
py_list = list(range(size))
np_arr = np.arange(size)

t = time.perf_counter()
_ = [x * 2 for x in py_list]
list_time = time.perf_counter() - t

t = time.perf_counter()
_ = np_arr * 2
np_time = time.perf_counter() - t

print(f"List : {list_time:.5f}s")
print(f"NumPy: {np_time:.5f}s")
print(f"NumPy is roughly {list_time / np_time:.0f}x faster here")

### Q2. Python list vs NumPy array: key differences?

| Feature | Python list | NumPy ndarray |
|---|---|---|
| Element types | Mixed allowed | One dtype for all elements |
| Memory layout | Array of pointers to objects | Contiguous raw values |
| `a * 2` | Repeats the list | Multiplies every element |
| `a + b` | Concatenates | Element-wise addition |
| Speed on numeric work | Slow | Fast (vectorized C) |
| Size | Can grow cheaply (`append`) | Fixed size; "growing" makes a new array |
| Dimensions | Nested lists, awkward | Native N-D support |

In [ ]:
lst = [1, 2, 3]
arr = np.array([1, 2, 3])

print("list * 2   :", lst * 2)
print("array * 2  :", arr * 2)
print("list + list:", lst + lst)
print("arr + arr  :", arr + arr)

# Memory comparison
big_list = list(range(1000))
big_arr = np.arange(1000)
list_bytes = sys.getsizeof(big_list) + sum(sys.getsizeof(i) for i in big_list)
print("\nlist  of 1000 ints:", list_bytes, "bytes")
print("array of 1000 ints:", big_arr.nbytes, "bytes")

### Q3. Why exactly is NumPy faster? (Interviewers love this one)

1. **Contiguous memory**: better CPU cache usage.
2. **Homogeneous dtype**: no type checking per element.
3. **Compiled C loops**: the loop runs outside the Python interpreter.
4. **SIMD / vectorized CPU instructions** and optimized BLAS libraries for linear algebra.
5. **Avoids Python object overhead**: a list of ints stores pointers to full Python `int` objects.

### Q4. What is a `dtype`? What happens when you mix types in one array? (upcasting)

**Answer:** `dtype` is the data type of the array's elements (`int64`, `float32`, `bool`, `<U5` for strings...). If you mix types, NumPy **upcasts** to the smallest type that can hold everything: bool < int < float < string.

Assigning a float into an int array **truncates silently** (no error).

In [ ]:
print(np.array([1, 2, 3]).dtype)
print(np.array([1, 2.5, 3]).dtype)        # int + float -> float64
print(np.array([1, True]).dtype)          # bool + int  -> int64
print(np.array([1, 2.5, "a"]).dtype)      # anything + str -> string

a = np.array([1, 2, 3])
a[1] = 2.9                                # silently truncated
print(a)

b = np.array([1.7, 2.2], dtype=np.float32)
print(b.astype(np.int64), b.astype(np.int64).dtype)   # astype returns a NEW array

---
# 2. Array Creation

### Q5. Different ways to create a NumPy array?

`np.array`, `np.zeros`, `np.ones`, `np.full`, `np.empty`, `np.eye`/`np.identity`, `np.arange`, `np.linspace`, `np.random.*`, `np.zeros_like`/`np.ones_like`/`np.full_like`, `np.fromfunction`, `np.loadtxt`.

In [ ]:
print(np.array([[1, 2], [3, 4]]))
print(np.zeros((2, 3)))
print(np.ones((2, 2), dtype=int))
print(np.full((2, 3), 7))
print(np.eye(3))
print(np.arange(0, 10, 2))
print(np.linspace(0, 1, 5))

template = np.array([[1, 2], [3, 4]])
print(np.zeros_like(template))
print(np.full_like(template, 9))

### Q6. `arange` vs `linspace`: when to use which?

- `arange(start, stop, step)`: you control the **step**. `stop` is excluded.
- `linspace(start, stop, num)`: you control the **number of points**. `stop` is **included** by default.

Gotcha: `arange` with a float step can give a surprising length because of floating-point error. Prefer `linspace` for floats.

In [ ]:
print(np.arange(0, 1, 0.1))
print(np.linspace(0, 1, 11))

# Floating point surprise: stop=1.3 is "excluded", yet we get 4 elements
# and the last one is ~1.3 (because (1.3 - 1) / 0.1 is slightly above 3 in floating point)
x = np.arange(1, 1.3, 0.1)
print(len(x), x[-1])

# linspace avoids this: you fix the count, so the length is always what you asked for
print(len(np.linspace(1, 1.3, 4)), np.linspace(1, 1.3, 4))

### Q7. `np.zeros` vs `np.empty`?

`np.zeros` fills memory with 0. `np.empty` just **allocates** memory without initializing, so it contains whatever garbage was there. `empty` is slightly faster but only safe when you will overwrite every element.

In [ ]:
e = np.empty((2, 2))
print(e.shape)   # values are unpredictable, never rely on them
print(np.zeros((2, 2)))

---
# 3. Array Attributes

### Q8. Explain `shape`, `ndim`, `size`, `dtype`, `itemsize`, `nbytes`, `strides`. How is `shape` different from `len()`?

- `shape`: tuple with the size of each dimension
- `ndim`: number of dimensions (axes)
- `size`: total number of elements
- `itemsize`: bytes per element
- `nbytes`: `size * itemsize`
- `strides`: bytes to step in each dimension to reach the next element
- `len(a)`: size of the **first** dimension only

In [ ]:
a = np.arange(24).reshape(2, 3, 4)
print("shape   :", a.shape)
print("ndim    :", a.ndim)
print("size    :", a.size)
print("dtype   :", a.dtype)
print("itemsize:", a.itemsize)
print("nbytes  :", a.nbytes)
print("strides :", a.strides)
print("len(a)  :", len(a))        # only first dimension

---
# 4. Indexing, Slicing, Masking, View vs Copy

### Q9. How does indexing and slicing work in 2D arrays?

Syntax: `a[row_selection, col_selection]`. Slices follow `start:stop:step`, negative indices count from the end, and `[::-1]` reverses.

In [ ]:
m = np.arange(1, 17).reshape(4, 4)
print(m, "\n")

print("m[1, 2]      :", m[1, 2])
print("m[:, 1]      :", m[:, 1])        # whole column 1
print("m[1]         :", m[1])           # whole row 1
print("m[1:3, 1:3]  :\n", m[1:3, 1:3])
print("m[::2, ::2]  :\n", m[::2, ::2])
print("m[-1]        :", m[-1])
print("m[::-1]      :\n", m[::-1])      # rows reversed

### Q10. What is boolean masking? Why can't you use `and` / `or` on arrays?

**Answer:** A boolean mask is an array of `True`/`False` of the same shape; `a[mask]` returns only the `True` positions. Combine conditions with `&`, `|`, `~` (with **parentheses**), because Python's `and`/`or` try to convert a whole array to a single bool and raise `ValueError: The truth value of an array ... is ambiguous`.

In [ ]:
a = np.array([5, 12, 7, 20, 3])
mask = a > 6
print(mask)
print(a[mask])
print(a[(a > 6) & (a < 15)])
print(a[~mask])

b = a.copy()
b[b > 10] = 0               # assignment through a mask
print(b)

try:
    a[(a > 6) and (a < 15)]
except ValueError as e:
    print("\nValueError:", e)

### Q11. What is fancy indexing? How is it different from slicing?

**Answer:** Fancy indexing means indexing with a list/array of integer positions. Unlike slicing, it can pick arbitrary, repeated, or reordered elements, and it **always returns a copy**. Slicing returns a **view**.

In [ ]:
a = np.array([10, 20, 30, 40, 50])
print(a[[0, 2, 4]])
print(a[[0, 0, 1]])        # repeats allowed

m = np.arange(9).reshape(3, 3)
print(m)
print(m[[0, 1, 2], [2, 1, 0]])          # picks (0,2), (1,1), (2,0): anti-diagonal
print(m[np.ix_([0, 2], [0, 2])])        # rows 0,2 x cols 0,2: the 'grid' selection

### Q12. View vs copy. What is the difference and how can you check?

- **View**: a new array object that **shares the same memory**. Changing one changes the other. Basic slicing, `reshape` (when possible), `ravel`, `.T` give views.
- **Copy**: new memory. Fancy indexing, boolean masking, `.copy()`, `flatten()` give copies.

Check with `np.shares_memory(a, b)` or `b.base is a`.

In [ ]:
a = np.arange(5)

v = a[1:4]            # slice -> view
v[0] = 99
print("after editing view:", a, "| shares memory:", np.shares_memory(a, v))

f = a[[1, 2, 3]]      # fancy index -> copy
f[0] = -1
print("after editing fancy copy:", a, "| shares memory:", np.shares_memory(a, f))

c = a[1:4].copy()     # explicit copy
print("explicit copy shares memory:", np.shares_memory(a, c))
print("v.base is a:", v.base is a)

# Plain assignment does NOT copy at all, it's the same object
b = a
b[0] = 1000
print(a, "| same object:", a is b)

### Q13. What are the uses of `np.where`?

1. `np.where(cond)` returns the **indices** where the condition is True.
2. `np.where(cond, x, y)` is a vectorized if-else: picks from `x` where True, else from `y`.

In [ ]:
a = np.array([1, 5, 8, 2, 9])
print(np.where(a > 4))                   # tuple of index arrays
print(np.where(a > 4, "big", "small"))
print(np.where(a > 4, a, 0))             # keep big values, zero the rest

---
# 5. Broadcasting (Top interview topic)

### Q14. What is broadcasting? What are the rules?

**Answer:** Broadcasting lets NumPy do element-wise operations on arrays of **different shapes** without copying data.

**Rules** (compare shapes from the **rightmost** dimension to the left):
1. If the arrays have different `ndim`, pad the smaller shape with 1s on the **left**.
2. Two dimensions are compatible if they are **equal** or one of them is **1**.
3. A dimension of size 1 is stretched to match the other.
4. If any dimension pair is incompatible -> `ValueError`.

In [ ]:
a = np.ones((3, 4))
b = np.arange(4)                       # shape (4,) -> treated as (1, 4)
print((a + b).shape)

col = np.arange(3).reshape(3, 1)       # (3, 1)
row = np.arange(4).reshape(1, 4)       # (1, 4)
print(col + row)                       # -> (3, 4) grid

print(np.broadcast_shapes((3, 4), (4,)))
print(np.broadcast_shapes((3, 1), (1, 4)))
print(np.broadcast_shapes((8, 1, 6, 1), (7, 1, 5)))   # classic example

try:
    np.ones((3, 4)) + np.ones(3)
except ValueError as e:
    print("\nValueError:", e)

### Q15. Which of these broadcast? `(3,4)+(4,)`, `(3,4)+(3,)`, `(3,1)+(1,4)`, `(2,3)+(3,2)`

| Shapes | Result |
|---|---|
| `(3,4)` + `(4,)` | OK -> `(3,4)` |
| `(3,4)` + `(3,)` | **Error**: trailing dims 4 vs 3 |
| `(3,1)` + `(1,4)` | OK -> `(3,4)` |
| `(2,3)` + `(3,2)` | **Error**: 3 vs 2 and 2 vs 3 |

Fix for the second: `a + b[:, np.newaxis]` makes `b` shape `(3,1)`.

### Q16. How do you subtract the row mean / column mean from a matrix?

Use `keepdims=True` (or `reshape`) so the mean keeps a dimension and broadcasts correctly.

In [ ]:
x = np.array([[1., 2., 3.],
              [4., 5., 6.]])

print("minus row mean:\n", x - x.mean(axis=1, keepdims=True))
print("minus col mean:\n", x - x.mean(axis=0))

---
# 6. Reshaping & Transposing

### Q17. How does `reshape` work? What does `-1` mean?

`reshape` changes the shape without changing data; total elements must match. `-1` means "figure this dimension out for me" (only one `-1` allowed). It returns a **view** when possible.

In [ ]:
a = np.arange(12)
print(a.reshape(3, 4).shape)
print(a.reshape(2, -1).shape)
print(a.reshape(-1, 6).shape)

r = a.reshape(3, 4)
r[0, 0] = 100
print("original changed too:", a[0])      # reshape returned a view

try:
    a.reshape(5, 3)
except ValueError as e:
    print("ValueError:", e)

### Q18. `flatten()` vs `ravel()`?

- `flatten()` always returns a **copy**.
- `ravel()` returns a **view** when possible (faster, but edits can affect the original).

In [ ]:
m = np.arange(6).reshape(2, 3)
f = m.flatten()
r = m.ravel()

f[0] = 99
print("after editing flatten:", m[0, 0])   # unchanged

r[0] = 77
print("after editing ravel  :", m[0, 0])   # changed

### Q19. How do you transpose? What about N-D arrays?

`a.T` or `np.transpose(a)` reverses axes. For N-D use `np.transpose(a, axes)` or `np.swapaxes`. Transpose returns a **view** (only strides change).

In [ ]:
m = np.arange(6).reshape(2, 3)
print(m.T.shape, np.transpose(m).shape, "| view:", m.T.base is m)

t = np.arange(24).reshape(2, 3, 4)
print(t.transpose(2, 0, 1).shape)
print(np.swapaxes(t, 0, 2).shape)

### Q20. What are `np.newaxis`, `expand_dims` and `squeeze`?

- `np.newaxis` (same as `None`) / `np.expand_dims`: add a dimension of size 1 (handy for broadcasting).
- `np.squeeze`: remove dimensions of size 1.

In [ ]:
a = np.array([1, 2, 3])
print(a.shape, a[:, np.newaxis].shape, a[np.newaxis, :].shape, np.expand_dims(a, 0).shape)

b = np.zeros((1, 3, 1))
print(np.squeeze(b).shape, np.squeeze(b, axis=0).shape)

### Q21. `reshape` vs `resize`?

`reshape` requires the element count to match. `np.resize(a, shape)` will **repeat** data to fill (or truncate) to the new size and returns a new array.

In [ ]:
a = np.arange(4)
print(np.resize(a, (2, 4)))     # repeats [0 1 2 3] to fill
print(np.resize(a, (1, 2)))     # truncates

---
# 7. Math, Aggregation & Ufuncs

### Q22. `*` vs `@` (or `np.dot`)?

- `*` is **element-wise** multiplication (needs broadcast-compatible shapes).
- `@` / `np.matmul` / `np.dot` is **matrix multiplication** (inner dimensions must match).

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print("A * B  (element-wise):\n", A * B)
print("A @ B  (matrix mult) :\n", A @ B)
print("np.dot(A, B)         :\n", np.dot(A, B))
print("A ** 2 (element-wise power):\n", A ** 2)

### Q23. What does `axis` mean in aggregations?

`axis` is the dimension that gets **collapsed**.
- `axis=0`: collapse rows -> one result **per column**
- `axis=1`: collapse columns -> one result **per row**
- `keepdims=True` keeps the collapsed axis as size 1 (useful for broadcasting).

In [ ]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])
print("sum all     :", m.sum())
print("sum axis=0  :", m.sum(axis=0))       # per column
print("sum axis=1  :", m.sum(axis=1))       # per row
print("keepdims    :", m.sum(axis=1, keepdims=True).shape)
print("mean axis=0 :", m.mean(axis=0))
print("max axis=1  :", m.max(axis=1))
print("argmax axis=1:", m.argmax(axis=1))

### Q24. What is a ufunc (universal function)?

**Answer:** A function that operates **element-wise** on arrays, implemented in C, supporting broadcasting and type casting. Examples: `np.add`, `np.sqrt`, `np.exp`, `np.sin`. Ufuncs have extra methods: `reduce`, `accumulate`, `outer`, and an optional `out=` argument to write into an existing array and save memory.

In [ ]:
a = np.array([1, 4, 9])
print(np.sqrt(a))
print(np.add(a, 1))
print(np.add.reduce(a))          # same as a.sum()
print(np.add.accumulate(a))      # same as cumsum
print(np.add.outer([1, 2], [10, 20]))

out = np.empty(3)
np.sqrt(a, out=out)              # result stored in preallocated array
print(out)

### Q25. Is `np.vectorize` actually faster?

**No.** `np.vectorize` is just a convenience wrapper around a Python `for` loop. For real speed, rewrite using native NumPy operations (`np.where`, boolean masks, ufuncs).

In [ ]:
def f(x):
    return x ** 2 if x % 2 == 0 else -x

vf = np.vectorize(f)
n = np.arange(6)
print(vf(n))                                         # convenient, but loops in Python
print(np.where(n % 2 == 0, n ** 2, -n))              # truly vectorized

### Q26. Important gotcha: `np.std` / `np.var` default `ddof`?

NumPy uses `ddof=0` (**population** std) by default. Pandas `.std()` uses `ddof=1` (**sample** std). For sample statistics in NumPy, pass `ddof=1`.

In [ ]:
x = np.array([2, 4, 4, 4, 5, 5, 7, 9])
print("population std (ddof=0):", np.std(x))
print("sample std     (ddof=1):", np.std(x, ddof=1))
print("variance:", np.var(x), "| median:", np.median(x))
print("percentiles 25/50/75:", np.percentile(x, [25, 50, 75]))

### Q27. `cumsum`, `cumprod`, `diff`: what do they do?

In [ ]:
a = np.array([1, 2, 3, 4])
print("cumsum :", np.cumsum(a))
print("cumprod:", np.cumprod(a))
print("diff   :", np.diff(a))
print("diff n=2:", np.diff(np.array([1, 4, 9, 16]), n=2))

### Q28. Rounding functions & the `np.round` surprise

`np.round` uses **banker's rounding** (round half to even): `0.5 -> 0`, `1.5 -> 2`, `2.5 -> 2`. Also know `floor`, `ceil`, `trunc`, `clip`.

In [ ]:
a = np.array([-1.5, -0.5, 0.5, 1.5, 2.5])
print("round:", np.round(a))
print("floor:", np.floor(a))
print("ceil :", np.ceil(a))
print("trunc:", np.trunc(a))
print("clip(-1, 1):", np.clip(a, -1, 1))

### Q29. What is integer overflow in NumPy?

Fixed-size integer dtypes **wrap around silently** (no error) when the value exceeds their range. Use a larger dtype (`int64`) or check limits with `np.iinfo` / `np.finfo`.

In [ ]:
a = np.array([127], dtype=np.int8)
print("int8 127 + 1 ->", a + 1)                 # wraps to -128
print("upcast first  ->", a.astype(np.int16) + 1)
print(np.iinfo(np.int8).min, np.iinfo(np.int8).max)
print("float32 eps:", np.finfo(np.float32).eps)

### Q30. Why not compare floats with `==`? What to use instead?

Floating-point representation errors (`0.1 + 0.2 != 0.3`). Use `np.isclose` (element-wise) or `np.allclose` (whole array). `np.array_equal` checks exact equality of shape and values. `any()` / `all()` reduce boolean arrays.

In [ ]:
print(0.1 + 0.2 == 0.3)
print(np.isclose(0.1 + 0.2, 0.3))
print(np.allclose([1, 2], [1.0000001, 2]))

a = np.array([1, 2, 3])
print((a > 2).any(), (a > 0).all(), np.array_equal(a, [1, 2, 3]))

---
# 8. Joining & Splitting

### Q31. `concatenate` vs `stack` vs `vstack` vs `hstack`?

- `np.concatenate(list, axis)`: joins along an **existing** axis.
- `np.stack(list, axis)`: joins along a **new** axis (all shapes must match).
- `np.vstack`: stack rows (axis 0), `np.hstack`: stack columns (axis 1 for 2D), `np.column_stack`: 1D arrays become columns.

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("concatenate:", np.concatenate([a, b]))
print("vstack:\n", np.vstack([a, b]))
print("hstack:", np.hstack([a, b]))
print("stack axis=0:", np.stack([a, b], axis=0).shape)
print("stack axis=1:", np.stack([a, b], axis=1).shape)
print("column_stack:\n", np.column_stack([a, b]))

m1, m2 = np.ones((2, 2)), np.zeros((2, 2))
print("concat 2D axis=1 shape:", np.concatenate([m1, m2], axis=1).shape)

### Q32. How do you split arrays?

`np.split` (needs equal division), `np.array_split` (allows uneven), `np.hsplit`, `np.vsplit`.

In [ ]:
print(np.split(np.arange(9), 3))
print(np.array_split(np.arange(10), 3))       # uneven is fine

m = np.arange(16).reshape(4, 4)
print(np.hsplit(m, 2)[0])
print(np.vsplit(m, 2)[1])

### Q33. `append`, `insert`, `delete`: are they in-place? Are they efficient?

None are in-place: each returns a **new array** (copying all data). Calling `np.append` in a loop is slow, so collect in a Python list and convert once, or preallocate.

In [ ]:
a = np.array([1, 2, 3])
print(np.append(a, 4), np.insert(a, 1, 99), np.delete(a, 0))
print("original untouched:", a)

m = np.arange(6).reshape(2, 3)
print(np.append(m, [[7, 8, 9]], axis=0))
print(np.delete(m, 1, axis=1))

---
# 9. Sorting, Searching & Set Operations

### Q34. `np.sort` vs `a.sort()`? How to sort descending?

- `np.sort(a)` returns a **sorted copy**.
- `a.sort()` sorts **in place** and returns `None`.
- Descending: `np.sort(a)[::-1]` or `-np.sort(-a)`.

In [ ]:
a = np.array([3, 1, 2])
print(np.sort(a), "| original:", a)
print("descending:", np.sort(a)[::-1])
a.sort()
print("after in-place sort:", a)

m = np.array([[3, 1], [2, 9], [1, 5]])
print(np.sort(m, axis=0))     # sort each column independently

### Q35. What is `argsort` and why is it useful?

`argsort` returns the **indices** that would sort the array. Use it to sort one array by another (e.g., sort rows by a column, sort names by scores).

In [ ]:
scores = np.array([30, 10, 20])
names = np.array(["Asha", "Ravi", "Meera"])

idx = np.argsort(scores)
print(idx, scores[idx])
print("names by score :", names[idx])
print("names descending:", names[np.argsort(-scores)])

data = np.array([[1, 300], [2, 100], [3, 200]])
print("rows sorted by column 1:\n", data[data[:, 1].argsort()])

### Q36. `argmax` / `argmin`: what do they return? How do you get a 2D position?

They return the **index** of the max/min (first occurrence on ties). Without `axis`, the index is into the **flattened** array; use `np.unravel_index` to convert to (row, col).

In [ ]:
m = np.array([[1, 9, 3],
              [7, 2, 8]])
print(np.argmax(m))
print(np.argmax(m, axis=0), np.argmax(m, axis=1))
print(np.unravel_index(np.argmax(m), m.shape))
print("tie -> first occurrence:", np.argmax([1, 5, 5, 2]))

### Q37. How do you find unique values and their counts?

In [ ]:
a = np.array([3, 1, 2, 3, 1, 3])
vals, counts = np.unique(a, return_counts=True)
print(vals, counts)
print("most frequent:", vals[counts.argmax()])

u, first_idx, inverse = np.unique(a, return_index=True, return_inverse=True)
print("first occurrence index:", first_idx)
print("inverse mapping       :", inverse)

### Q38. What is `np.searchsorted`?

Binary search on a **sorted** array: returns the insertion index that keeps the order. `side='left'`/`'right'` decides placement among equal values. Much faster than scanning.

In [ ]:
a = np.array([10, 20, 30, 40])
print(np.searchsorted(a, 25))
print(np.searchsorted(a, 20, side="left"), np.searchsorted(a, 20, side="right"))

### Q39. Set operations & membership testing

In [ ]:
a = np.array([1, 2, 3, 4])
b = np.array([3, 4, 5])
print("isin       :", np.isin(a, b))
print("intersect1d:", np.intersect1d(a, b))
print("union1d    :", np.union1d(a, b))
print("setdiff1d  :", np.setdiff1d(a, b))
print("setxor1d   :", np.setxor1d(a, b))

z = np.array([0, 3, 0, 5])
print("nonzero:", np.nonzero(z), "| count_nonzero:", np.count_nonzero(z))

---
# 10. Linear Algebra

### Q40. `np.dot` vs `np.matmul` vs `@` vs `inner` vs `outer`?

- 1D · 1D: `dot` = inner product (a scalar).
- 2D · 2D: `dot` = `matmul` = `@` = matrix multiplication.
- N-D (>2): `matmul` treats leading dims as **batch** dims; `dot` behaves differently (sums over last axis of `a` and second-to-last of `b`). Prefer `@`.
- `np.outer(a, b)`: every pair product, shape `(len(a), len(b))`.

In [ ]:
A = np.array([[1, 2], [3, 4]])
v = np.array([1, 2])

print("A @ v      :", A @ v)
print("dot(v, v)  :", np.dot(v, v))
print("inner(v, v):", np.inner(v, v))
print("outer(v, v):\n", np.outer(v, v))
print("matmul(A,A):\n", np.matmul(A, A))

### Q41. Inverse, determinant, solving linear systems. `solve` vs `inv`?

For `Ax = b`, prefer `np.linalg.solve(A, b)`: it is faster and numerically more stable than computing `inv(A) @ b`. A **singular** matrix (det = 0) has no inverse and raises `LinAlgError`.

In [ ]:
A = np.array([[2., 1.],
              [1., 3.]])
b = np.array([3., 5.])

print("inverse:\n", np.linalg.inv(A))
print("det    :", np.linalg.det(A))

x = np.linalg.solve(A, b)
print("solution x:", x, "| A@x == b ?", np.allclose(A @ x, b))
print("A @ inv(A) = I ?", np.allclose(A @ np.linalg.inv(A), np.eye(2)))

S = np.array([[1, 2], [2, 4]])           # singular: row2 = 2 * row1
print("det of singular:", np.linalg.det(S))
try:
    np.linalg.inv(S)
except np.linalg.LinAlgError as e:
    print("LinAlgError:", e)

### Q42. Eigenvalues, rank, trace, norm, SVD (know the one-line meaning)

- **Eigen**: `A v = λ v`. `np.linalg.eig` gives eigenvalues `w` and eigenvectors (columns of `v`). Core of PCA.
- **Rank**: number of linearly independent rows/columns.
- **Trace**: sum of the diagonal.
- **Norm**: vector length / matrix magnitude.
- **SVD**: `A = U Σ Vᵀ`, used in dimensionality reduction and recommender systems.

In [ ]:
A = np.array([[2, 0], [0, 3]])
w, v = np.linalg.eig(A)
print("eigenvalues :", w)
print("eigenvectors:\n", v)
print("rank:", np.linalg.matrix_rank(A), "| trace:", np.trace(A))
print("norm of [3,4]:", np.linalg.norm([3, 4]))

U, S, Vt = np.linalg.svd(np.array([[1, 2], [3, 4]]))
print("singular values:", S)

---
# 11. Random Numbers

### Q43. How do you generate random numbers? Why set a seed? Legacy API vs `default_rng`?

Setting a **seed** makes results **reproducible**. The legacy API is `np.random.seed(...)` plus global functions; the modern recommended API is `rng = np.random.default_rng(seed)`.

| Need | Legacy | Modern `Generator` |
|---|---|---|
| Uniform [0, 1) | `np.random.rand(n)` | `rng.random(n)` |
| Standard normal | `np.random.randn(n)` | `rng.standard_normal(n)` |
| Random ints | `np.random.randint(a, b, n)` | `rng.integers(a, b, n)` |
| Pick from array | `np.random.choice` | `rng.choice` |

In [ ]:
np.random.seed(42)
print("legacy:", np.random.rand(3))

rng = np.random.default_rng(42)
print("rng.random   :", rng.random(3))
print("rng.integers :", rng.integers(0, 10, 5))
print("rng.normal   :", rng.normal(0, 1, 3))
print("rng.choice   :", rng.choice([1, 2, 3, 4], 2, replace=False))

### Q44. `shuffle` vs `permutation`?

`shuffle` shuffles **in place** (returns `None`). `permutation` returns a **shuffled copy** (or a permuted `arange(n)` if given an int).

In [ ]:
rng = np.random.default_rng(0)
a = np.arange(5)
p = rng.permutation(a)
print("original:", a, "| permutation copy:", p)
rng.shuffle(a)
print("after in-place shuffle:", a)

---
# 12. NaN & inf Handling

### Q45. How does NumPy treat `NaN`? How do you handle missing values?

- `NaN` is a **float** (`np.nan`), so integer arrays cannot hold it.
- `np.nan == np.nan` is **False**; use `np.isnan`.
- Any arithmetic with NaN gives NaN, so `a.sum()` becomes NaN. Use `nansum`, `nanmean`, `nanmax`, `nanstd`, ...
- `np.nan` is not `None`.

In [ ]:
a = np.array([1, np.nan, 3])
print(np.nan == np.nan, "|", np.isnan(a))
print("sum:", a.sum(), "| nansum:", np.nansum(a), "| nanmean:", np.nanmean(a), "| nanmax:", np.nanmax(a))
print("type(np.nan):", type(np.nan), "| np.nan is None:", np.nan is None)

print("drop NaN:", a[~np.isnan(a)])
filled = a.copy()
filled[np.isnan(filled)] = np.nanmean(a)
print("fill NaN with mean:", filled)

try:
    ints = np.array([1, 2, 3])
    ints[0] = np.nan
except ValueError as e:
    print("ValueError:", e)

with np.errstate(divide="ignore"):
    print("1/0.0 ->", 1 / np.array([0.0]), "| isinf:", np.isinf(np.array([1, np.inf])))

---
# 13. Memory, Performance & I/O

### Q46. Row-major (C) vs column-major (Fortran) order? What are strides?

NumPy defaults to **C order** (row-major): consecutive elements of a row sit next to each other in memory. **Strides** are the byte steps per axis. Transposing swaps strides without moving data, so `.T` of a C-contiguous array is F-contiguous. Iterating in memory order is faster (cache-friendly).

In [ ]:
c = np.arange(6).reshape(2, 3)
f = np.asfortranarray(c)
print("C strides:", c.strides, "| F strides:", f.strides)
print("c C-contiguous:", c.flags["C_CONTIGUOUS"], "| c.T F-contiguous:", c.T.flags["F_CONTIGUOUS"])

### Q47. How do you speed up NumPy code?

1. Replace Python loops with **vectorized** operations.
2. Use **broadcasting** instead of tiling/repeating data.
3. Use the smallest suitable **dtype** (`float32`, `int16`) to save memory.
4. Preallocate arrays (`np.empty`) instead of repeatedly appending.
5. Use `out=` / in-place ops (`a += 1`) to avoid temporaries.
6. Avoid unnecessary copies (know view vs copy).
7. Use `np.einsum`, `@`, BLAS-backed functions for heavy math.

In [ ]:
x = np.random.rand(1_000_000)

t = time.perf_counter()
s = 0.0
for v in x:
    s += v * v
loop_time = time.perf_counter() - t

t = time.perf_counter()
s2 = np.dot(x, x)
vec_time = time.perf_counter() - t

print(f"loop      : {loop_time:.4f}s")
print(f"vectorized: {vec_time:.6f}s  (~{loop_time / vec_time:.0f}x faster)")
print("same result:", np.isclose(s, s2))

# dtype saves memory
print("float64:", np.zeros(1_000_000, dtype=np.float64).nbytes / 1e6, "MB")
print("float32:", np.zeros(1_000_000, dtype=np.float32).nbytes / 1e6, "MB")

### Q48. `np.array` vs `np.asarray`? `copy` vs `deepcopy`?

- `np.array(a)` **copies** by default; `np.asarray(a)` does **not** copy if `a` is already an ndarray of the right dtype.
- `a.copy()` makes an independent copy of a numeric array. For arrays of **Python objects** (`dtype=object`) it is shallow, so use `copy.deepcopy` for nested objects.

In [ ]:
a = np.array([1, 2, 3])
print("asarray returns same object:", np.asarray(a) is a)
print("array returns same object  :", np.array(a) is a)

### Q49. How do you save/load arrays and read CSV data?

- `np.save` / `np.load` (binary `.npy`, fastest, keeps dtype & shape)
- `np.savez` for multiple arrays
- `np.savetxt` / `np.loadtxt` / `np.genfromtxt` (text/CSV; `genfromtxt` handles missing values)

In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmp:
    npy = os.path.join(tmp, "arr.npy")
    csv = os.path.join(tmp, "arr.csv")

    np.save(npy, np.arange(5))
    print("np.load :", np.load(npy))

    np.savetxt(csv, np.array([[1, 2], [3, 4]]), delimiter=",", fmt="%d")
    print("loadtxt :\n", np.loadtxt(csv, delimiter=","))

---
# 14. Predict the Output (Tricky Questions)

Try to guess the output **before** running each cell. Answers are at the end of this section.

**P1**

In [ ]:
a = np.arange(5)
b = a[1:4]
b[0] = 99
print(a)

**P2**

In [ ]:
print(0.1 + 0.2 == 0.3, np.nan == np.nan, np.nan is np.nan)

**P3**

In [ ]:
a = np.array([1, 2, 3])
a[1] = 2.9
print(a)

**P4**

In [ ]:
print(np.array([True, False, True]).sum(), np.array([1, 2, 3]) > 1)

**P5**

In [ ]:
a = np.arange(10)
print(a[::-3])

**P6**

In [ ]:
a = np.arange(6).reshape(2, 3)
print(a[:, 1].shape, a[:, [1]].shape, a[:, 1:2].shape)

**P7**

In [ ]:
print((np.arange(3) + np.arange(3).reshape(3, 1)).shape)

**P8**

In [ ]:
a = np.arange(5)
a[a > 2] = 0
print(a)

**P9**

In [ ]:
x = np.array([1, 2, 3])
y = x
y += 1
z = x + 1
print(x, z)

**P10**

In [ ]:
print(np.round([0.5, 1.5, 2.5, 3.5]))

### Answers

| # | Output | Why |
|---|---|---|
| P1 | `[ 0 99  2  3  4]` | Slice is a **view**, so editing `b` edits `a` |
| P2 | `False False True` | Float error; NaN never equals NaN; `np.nan` is a single object so `is` works |
| P3 | `[1 2 3]` | Float assigned into int array is **truncated** |
| P4 | `2 [False  True  True]` | `True` counts as 1; comparison is element-wise |
| P5 | `[9 6 3 0]` | Negative step starts from the end |
| P6 | `(2,) (2, 1) (2, 1)` | Integer index drops the dim; list index and slice keep it |
| P7 | `(3, 3)` | `(3,)` + `(3,1)` broadcasts to a grid |
| P8 | `[0 1 2 0 0]` | Boolean mask assignment |
| P9 | `[2 3 4] [3 4 5]` | `y = x` is the same object and `+=` is **in-place**; `x + 1` creates a new array |
| P10 | `[0. 2. 2. 4.]` | Banker's rounding (round half to even) |

---
# 15. Coding Challenges with Solutions

Each challenge is a typical interview task. Try writing your own solution first.

### C1. Normalize an array: Min-Max scaling and Z-score standardization

In [ ]:
x = np.array([10, 20, 30, 40, 50], dtype=float)
minmax = (x - x.min()) / (x.max() - x.min())
zscore = (x - x.mean()) / x.std()
print("min-max:", minmax)
print("z-score:", zscore.round(3))

### C2. Moving average with window size 3 (no loops)

In [ ]:
a = np.arange(1, 11)
w = 3

# Method 1: convolution
print(np.convolve(a, np.ones(w) / w, mode="valid"))

# Method 2: cumulative sum trick
cs = np.cumsum(np.insert(a, 0, 0))
print((cs[w:] - cs[:-w]) / w)

# Method 3: sliding window view
print(np.lib.stride_tricks.sliding_window_view(a, w).mean(axis=1))

### C3. One-hot encode integer labels

In [ ]:
labels = np.array([0, 2, 1, 2])
print(np.eye(3, dtype=int)[labels])

### C4. Detect outliers with the IQR rule and replace them with the median

In [ ]:
a = np.array([10, 12, 11, 500, 13, -300, 12])
q1, q3 = np.percentile(a, [25, 75])
iqr = q3 - q1
mask = (a < q1 - 1.5 * iqr) | (a > q3 + 1.5 * iqr)

cleaned = a.copy()
cleaned[mask] = np.median(a)
print("outlier mask:", mask)
print("cleaned     :", cleaned)

### C5. Find the second largest *distinct* element

In [ ]:
a = np.array([4, 9, 9, 2, 7])
print(np.unique(a)[-2])

### C6. Most frequent value in an array of non-negative ints

In [ ]:
a = np.array([1, 3, 3, 2, 3, 1])
print(np.bincount(a).argmax())

### C7. Rotate a matrix 90° clockwise

In [ ]:
m = np.arange(9).reshape(3, 3)
print(np.rot90(m, -1))
print(m.T[:, ::-1])         # transpose + reverse columns gives the same result

### C8. Pairwise Euclidean distance matrix using broadcasting (no loops)

In [ ]:
P = np.array([[0, 0], [3, 4], [6, 8]], dtype=float)
diff = P[:, None, :] - P[None, :, :]          # (3,1,2) - (1,3,2) -> (3,3,2)
D = np.sqrt((diff ** 2).sum(axis=-1))
print(D)

### C9. Create an 8x8 checkerboard pattern

In [ ]:
cb = np.zeros((8, 8), dtype=int)
cb[1::2, ::2] = 1
cb[::2, 1::2] = 1
print(cb)

### C10. Remove rows that contain any NaN

In [ ]:
m = np.array([[1, 2], [np.nan, 3], [4, 5]])
print(m[~np.isnan(m).any(axis=1)])

### C11. Replace NaNs with the column mean

In [ ]:
m = np.array([[1, np.nan], [3, 4], [np.nan, 8.0]])
col_mean = np.nanmean(m, axis=0)
rows, cols = np.where(np.isnan(m))
m[rows, cols] = col_mean[cols]
print(m)

### C12. Indices of the top-k largest elements

In [ ]:
a = np.array([5, 1, 9, 3, 7])
k = 2
top = np.argpartition(a, -k)[-k:]              # O(n), unordered
top = top[np.argsort(-a[top])]                 # order them largest first
print(top, a[top])

### C13. Check whether an array is sorted; count elements in a range

In [ ]:
a = np.array([1, 3, 5, 7, 9])
print("is sorted ascending:", np.all(a[:-1] <= a[1:]))
print("count 3 < x < 8    :", ((a > 3) & (a < 8)).sum())

### C14. Diagonal, trace, upper/lower triangle, anti-diagonal

In [ ]:
m = np.arange(1, 10).reshape(3, 3)
print("diag      :", np.diag(m))
print("trace     :", np.trace(m))
print("anti-diag :", np.diag(np.fliplr(m)))
print("upper:\n", np.triu(m))
print("lower:\n", np.tril(m))

### C15. Correlation and covariance between two arrays

In [ ]:
x = np.array([1, 2, 3, 4, 5])
y = np.array([2, 4, 5, 4, 5])
print("corr:", np.corrcoef(x, y)[0, 1].round(4))
print("cov matrix:\n", np.cov(x, y))

### C16. Swap two columns of a matrix

In [ ]:
m = np.arange(9).reshape(3, 3)
m2 = m.copy()
m2[:, [0, 2]] = m2[:, [2, 0]]
print(m2)

### C17. Matrix multiplication, trace and transpose with `einsum`

In [ ]:
A = np.arange(6).reshape(2, 3)
B = np.arange(6).reshape(3, 2)
print(np.einsum("ij,jk->ik", A, B))        # matrix multiplication
print(np.einsum("ii", np.eye(3)))          # trace
print(np.einsum("ij->ji", A))              # transpose

### C18. Remove duplicates while preserving the original order

In [ ]:
a = np.array([3, 1, 3, 2, 1])
_, first_idx = np.unique(a, return_index=True)
print(a[np.sort(first_idx)])

### C19. Replace all negative values with 0 (without a loop)

In [ ]:
a = np.array([-3, 5, -1, 8])
print(np.maximum(a, 0))
print(np.where(a < 0, 0, a))

### C20. Find positions where two arrays match

In [ ]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([1, 0, 3, 0, 5])
print(np.where(a == b)[0])

---
# 16. Rapid-Fire Theory Cheat Sheet

Quick one-liners to revise the night before the interview.

| Question | One-line answer |
|---|---|
| What is an `ndarray`? | Homogeneous, fixed-size, N-dimensional array in contiguous memory |
| Can a NumPy array hold mixed types? | Only via `dtype=object` or structured arrays (loses speed) |
| `np.array` vs `np.asarray`? | `array` copies by default, `asarray` avoids copying existing ndarrays |
| View vs copy? | View shares memory (slicing, reshape, ravel, `.T`); copy doesn't (fancy/boolean indexing, `.copy()`, `flatten`) |
| `flatten` vs `ravel`? | `flatten` always copies, `ravel` returns a view if possible |
| What is broadcasting? | Rule-based stretching of size-1 dims so different shapes work together |
| `*` vs `@`? | Element-wise vs matrix multiplication |
| Meaning of `axis=0`/`axis=1`? | Collapse rows (per-column result) / collapse columns (per-row result) |
| `np.nan == np.nan`? | `False`; use `np.isnan` |
| `np.nan` vs `None`? | `nan` is a float; `None` is a Python object |
| Default `ddof` in `np.std`? | 0 (population); pandas uses 1 |
| `np.sum` vs built-in `sum`? | `np.sum` is much faster on arrays and supports `axis` |
| What does `-1` do in `reshape`? | Infers that dimension automatically |
| What is a ufunc? | Vectorized element-wise C function with `reduce`/`accumulate`/`outer` methods |
| Is `np.vectorize` fast? | No, it's a loop wrapper for convenience |
| `np.dot` vs `np.matmul`? | Same for 1D/2D; differ for N-D (`matmul` broadcasts batches) |
| `solve` vs `inv`? | `solve` is faster and more stable |
| `sort` vs `argsort`? | Sorted values vs indices that sort |
| `concatenate` vs `stack`? | Existing axis vs new axis |
| `arange` vs `linspace`? | Step-based (stop excluded) vs count-based (stop included) |
| `np.round(2.5)`? | `2.0` (banker's rounding) |
| Integer overflow? | Fixed-width ints wrap silently; upcast or check `np.iinfo` |
| How to make results reproducible? | `np.random.seed(n)` or `np.random.default_rng(n)` |
| `np.matrix` vs `ndarray`? | `np.matrix` is deprecated; always use `ndarray` |
| How to reduce memory? | Smaller dtype, avoid copies, in-place ops, `out=` |
| Masked arrays? | `np.ma` lets you ignore invalid entries in computations |

---
### Done with NumPy? Next stop: Pandas

Good bridge topics: `Series` = labeled 1D ndarray, `DataFrame` = labeled 2D table built on NumPy; boolean masking, `axis`, broadcasting, `NaN` handling, `ddof` differences and vectorization all carry over directly.